## Objetivo

Realizar la limpieza del conjunto de datos **Evaluaciones Ambientales de Seguimiento (EAS) - Aire**
del OEFA a partir de los problemas identificados durante la etapa de exploración.

Las principales decisiones de limpieza son:

- Trabajar con registros correspondientes al periodo 2022-2024.
- Utilizar mediciones con periodicidad de una hora.
- Seleccionar los contaminantes PM10, PM2.5, SO2, NO2 y CO.
- Conservar únicamente valores cuantitativos válidos.
- Homogeneizar las unidades µg/m³ y μg/m³.
- Excluir registros expresados en ppb para evitar mezclar escalas diferentes.
- Eliminar concentraciones negativas.
- Estandarizar los nombres de las variables y contaminantes.
- Revisar y eliminar únicamente registros exactamente duplicados.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!pip install duckdb -q

In [ ]:
import os
import duckdb
import pandas as pd

ARCHIVO = "/content/drive/MyDrive/Concurrente/EAS_Aire_0.csv"

CARPETA_SALIDA = "/content/drive/MyDrive/Concurrente/Procesado"

os.makedirs(CARPETA_SALIDA, exist_ok=True)

con = duckdb.connect()

## 2. Carga del dataset original

In [ ]:
con.execute(f"""
CREATE OR REPLACE VIEW eas_original AS

SELECT *
FROM read_csv_auto(
    '{ARCHIVO}',
    sample_size=100000
);
""")

print("Vista 'eas_original' creada correctamente.")

Vista 'eas_original' creada correctamente.


## 3. Verificación inicial

In [ ]:
TOTAL_ORIGINAL = con.execute("""
SELECT COUNT(*)
FROM eas_original
""").fetchone()[0]

print(f"Registros originales: {TOTAL_ORIGINAL:,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Registros originales: 11,407,394


## 4. Estandarización inicial de variables

Se crea una representación intermedia del dataset con nombres de columnas más simples
y tipos de datos adecuados.

La fecha almacenada en formato serial de Excel se transforma a una fecha convencional.
La hora se convierte al tipo temporal correspondiente y la columna `Valor` se intenta
convertir a un valor numérico.

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW etapa_base AS

SELECT
    TRIM("Número.de.informe") AS numero_informe,
    TRIM("Nombre.de.la.Evaluación") AS nombre_evaluacion,

    TRIM("Nombre.del.punto") AS punto,

    TRY_CAST("Este" AS DOUBLE) AS este,
    TRY_CAST("Norte" AS DOUBLE) AS norte,
    TRY_CAST("Altitud" AS DOUBLE) AS altitud,
    TRY_CAST("Zona" AS INTEGER) AS zona,

    TRIM("Datum") AS datum,

    TRIM("Tipo.de.análisis") AS tipo_analisis,
    TRIM("Periodo") AS periodo,

    DATE '1899-12-30'
        + CAST(
            TRY_CAST("Fecha.inicio" AS DOUBLE)
            AS INTEGER
        ) AS fecha,

    TRY_CAST(
        NULLIF(TRIM("Hora.inicio"), '-')
        AS TIME
    ) AS hora,

    TRIM("Parámetro") AS parametro_original,

    TRIM("Unidad.de.medida") AS unidad_original,

    TRY_CAST(
        REPLACE(TRIM("Valor"), ',', '.')
        AS DOUBLE
    ) AS valor

FROM eas_original;
""")

print("Estandarización inicial completada.")

Estandarización inicial completada.


## 5. Selección del periodo reciente

El dataset original contiene observaciones desde 2018 hasta 2024.

Para trabajar con información reciente se conservan únicamente los registros comprendidos entre el 1 de enero de 2022 y el 31 de diciembre de 2024.

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW etapa_fecha AS

SELECT *
FROM etapa_base

WHERE fecha >= DATE '2022-01-01'
  AND fecha < DATE '2025-01-01';
""")

In [ ]:
TOTAL_FECHA = con.execute("""
SELECT COUNT(*)
FROM etapa_fecha
""").fetchone()[0]

print(f"Antes del filtro: {TOTAL_ORIGINAL:,}")
print(f"Después del filtro 2022-2024: {TOTAL_FECHA:,}")
print(f"Registros eliminados: {TOTAL_ORIGINAL - TOTAL_FECHA:,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Antes del filtro: 11,407,394
Después del filtro 2022-2024: 9,138,388
Registros eliminados: 2,269,006


## 6. Homogeneización de los nombres de los contaminantes

- El dataset contiene diferentes formas de representar algunos parámetros ambientales.

- Se clasifican las mediciones correspondientes a los cinco contaminantes seleccionados: PM10, PM2.5, SO2, NO2 y CO.

- Los demás parámetros ambientales se mantienen fuera del conjunto destinado al caso de uso.

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW etapa_contaminantes AS

SELECT
    *,

    CASE

        -- PM2.5 se evalúa antes que PM10 para evitar clasificaciones incorrectas.
        WHEN
            LOWER(parametro_original) LIKE '%pm2,5%'
            OR LOWER(parametro_original) LIKE '%pm2.5%'
            OR LOWER(parametro_original) LIKE '%2,5%'
            OR LOWER(parametro_original) LIKE '%2.5%'
        THEN 'PM2.5'

        WHEN (
            LOWER(parametro_original) LIKE '%pm10%'
            OR (
                LOWER(parametro_original) LIKE '%particul%'
                AND LOWER(parametro_original) LIKE '%10%'
            )
        )
        THEN 'PM10'

        WHEN LOWER(parametro_original)
             LIKE '%dióxido de azufre%'
        THEN 'SO2'

        WHEN LOWER(parametro_original)
             LIKE '%dióxido de nitrógeno%'
        THEN 'NO2'

        WHEN LOWER(parametro_original)
             LIKE '%monóxido de carbono%'
        THEN 'CO'

        ELSE NULL

    END AS contaminante

FROM etapa_fecha;
""")

In [ ]:
resumen_clasificacion = con.execute("""
SELECT
    contaminante,
    COUNT(*) AS registros
FROM etapa_contaminantes
WHERE contaminante IS NOT NULL
GROUP BY contaminante
ORDER BY registros DESC
""").df()

display(resumen_clasificacion)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,contaminante,registros
0,SO2,936186
1,PM10,735162
2,PM2.5,634747
3,CO,429528
4,NO2,260806


## 7. Selección de contaminantes

Se eliminan del conjunto de trabajo los parámetros que no corresponden a PM10, PM2.5,
SO2, NO2 o CO.

Esta selección responde al caso de uso planteado para analizar patrones de contaminación atmosférica.

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW etapa_parametros AS

SELECT *
FROM etapa_contaminantes

WHERE contaminante IS NOT NULL;
""")

In [ ]:
TOTAL_PARAMETROS = con.execute("""
SELECT COUNT(*)
FROM etapa_parametros
""").fetchone()[0]

print(f"Registros con contaminantes seleccionados: {TOTAL_PARAMETROS:,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Registros con contaminantes seleccionados: 2,996,429


## 8. Eliminación de valores no cuantificables

Durante la exploración se identificaron valores como `-`, `NA`, `N.D.`, `<LC`, `<L.D.`,
códigos alfabéticos y otras representaciones que no corresponden directamente a una
concentración numérica.

Para el conjunto destinado al modelo se conservan únicamente las observaciones que
pueden interpretarse directamente como valores cuantitativos.

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW etapa_numericos AS

SELECT *
FROM etapa_parametros

WHERE valor IS NOT NULL;
""")

In [ ]:
TOTAL_NUMERICOS = con.execute("""
SELECT COUNT(*)
FROM etapa_numericos
""").fetchone()[0]

print(f"Antes del filtro numérico: {TOTAL_PARAMETROS:,}")
print(f"Valores numéricos válidos: {TOTAL_NUMERICOS:,}")
print(f"No numéricos excluidos: {TOTAL_PARAMETROS - TOTAL_NUMERICOS:,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Antes del filtro numérico: 2,996,429
Valores numéricos válidos: 1,994,760
No numéricos excluidos: 1,001,669


## 9. Homogeneización de la periodicidad

El dataset contiene mediciones realizadas en periodos diferentes, como una hora,
tres horas móviles, ocho horas móviles y veinticuatro horas.

Para evitar mezclar observaciones con distinta resolución temporal se seleccionan
únicamente mediciones correspondientes a un periodo de una hora.

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW etapa_1hora AS

SELECT *
FROM etapa_numericos

WHERE LOWER(TRIM(periodo)) = '1 hora';
""")

In [ ]:
TOTAL_1H = con.execute("""
SELECT COUNT(*)
FROM etapa_1hora
""").fetchone()[0]

print(f"Registros numéricos antes del filtro: {TOTAL_NUMERICOS:,}")
print(f"Mediciones de 1 hora: {TOTAL_1H:,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Registros numéricos antes del filtro: 1,994,760
Mediciones de 1 hora: 1,670,191


## 10. Homogeneización de unidades de medida

Se identificaron dos representaciones visualmente similares de microgramos por metro
cúbico: `µg/m³` y `μg/m³`.

Ambas representan la misma unidad y son estandarizadas como `µg/m³`.

Las mediciones expresadas en `ppb` no se convierten, ya que esto requeriría
conversiones específicas para cada compuesto y condiciones físicas adicionales.

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW etapa_unidades AS

SELECT
    *,

    CASE
        WHEN TRIM(unidad_original) IN ('µg/m³', 'μg/m³')
        THEN 'µg/m³'

        ELSE TRIM(unidad_original)
    END AS unidad

FROM etapa_1hora;
""")

In [ ]:
unidades = con.execute("""
SELECT
    contaminante,
    unidad,
    COUNT(*) AS registros

FROM etapa_unidades

GROUP BY contaminante, unidad
ORDER BY contaminante, registros DESC
""").df()

display(unidades)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,contaminante,unidad,registros
0,CO,µg/m³,77114
1,CO,ppb,77114
2,NO2,µg/m³,85094
3,NO2,ppb,85094
4,PM10,µg/m³,466808
5,PM2.5,µg/m³,423496
6,SO2,µg/m³,240664
7,SO2,ppb,214807


## 11. Selección de una unidad común

Para garantizar que las concentraciones utilizadas en el conjunto final se encuentren
expresadas mediante una misma unidad física, se conservan únicamente las mediciones
registradas en microgramos por metro cúbico (`µg/m³`).

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW etapa_microgramos AS

SELECT *
FROM etapa_unidades

WHERE unidad = 'µg/m³';
""")

In [ ]:
TOTAL_MICROGRAMOS = con.execute("""
SELECT COUNT(*)
FROM etapa_microgramos
""").fetchone()[0]

print(f"Mediciones horarias antes del filtro de unidad: {TOTAL_1H:,}")
print(f"Mediciones en µg/m³: {TOTAL_MICROGRAMOS:,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Mediciones horarias antes del filtro de unidad: 1,670,191
Mediciones en µg/m³: 1,293,176


## 12. Validación de las concentraciones

Las concentraciones de contaminantes atmosféricos no deben presentar valores negativos.

Se analiza si existen concentraciones menores que cero antes de eliminarlas.
Los valores elevados no se eliminan automáticamente porque podrían representar eventos
ambientales reales y no necesariamente errores.

In [ ]:
valores_negativos = con.execute("""
SELECT
    contaminante,
    COUNT(*) AS negativos,
    MIN(valor) AS valor_minimo

FROM etapa_microgramos

WHERE valor < 0

GROUP BY contaminante
ORDER BY negativos DESC
""").df()

display(valores_negativos)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,contaminante,negativos,valor_minimo


In [ ]:
con.execute("""
CREATE OR REPLACE VIEW etapa_valores_validos AS

SELECT *
FROM etapa_microgramos

WHERE valor >= 0;
""")

print("Vista 'etapa_valores_validos' creada correctamente.")

Vista 'etapa_valores_validos' creada correctamente.


In [ ]:
TOTAL_VALIDOS = con.execute("""
SELECT COUNT(*)
FROM etapa_valores_validos
""").fetchone()[0]

print(f"Antes de validar concentraciones: {TOTAL_MICROGRAMOS:,}")
print(f"Después de eliminar valores negativos: {TOTAL_VALIDOS:,}")
print(f"Valores negativos eliminados: {TOTAL_MICROGRAMOS - TOTAL_VALIDOS:,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Antes de validar concentraciones: 1,293,176
Después de eliminar valores negativos: 1,293,176
Valores negativos eliminados: 0


## 13. Validación de información espacial

El caso de uso considera patrones espaciales, por lo que se verifica que las observaciones
dispongan de punto de monitoreo y coordenadas válidas.

No se eliminan valores de altitud iguales a cero, ya que pueden ser válidos en zonas
ubicadas cerca del nivel del mar.

In [ ]:
problemas_espaciales = con.execute("""
SELECT
    COUNT(*) FILTER (
        WHERE punto IS NULL
           OR TRIM(punto) = ''
           OR TRIM(punto) = '-'
    ) AS punto_invalido,

    COUNT(*) FILTER (
        WHERE este IS NULL OR este <= 0
    ) AS este_invalido,

    COUNT(*) FILTER (
        WHERE norte IS NULL OR norte <= 0
    ) AS norte_invalido,

    COUNT(*) FILTER (
        WHERE zona IS NULL
    ) AS zona_invalida

FROM etapa_valores_validos;
""").df()

display(problemas_espaciales)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,punto_invalido,este_invalido,norte_invalido,zona_invalida
0,0,0,0,0


In [ ]:
con.execute("""
CREATE OR REPLACE VIEW etapa_espacial AS

SELECT *
FROM etapa_valores_validos

WHERE punto IS NOT NULL
  AND TRIM(punto) <> ''
  AND TRIM(punto) <> '-'

  AND este IS NOT NULL
  AND este > 0

  AND norte IS NOT NULL
  AND norte > 0

  AND zona IS NOT NULL;
""")

In [ ]:
TOTAL_ESPACIAL = con.execute("""
SELECT COUNT(*)
FROM etapa_espacial
""").fetchone()[0]

print(f"Antes de validar ubicación: {TOTAL_VALIDOS:,}")
print(f"Después de validar ubicación: {TOTAL_ESPACIAL:,}")
print(f"Registros eliminados: {TOTAL_VALIDOS - TOTAL_ESPACIAL:,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Antes de validar ubicación: 1,293,176
Después de validar ubicación: 1,293,176
Registros eliminados: 0


## 14. Identificación de registros duplicados

Se buscan observaciones exactamente repetidas considerando la información necesaria
para identificar una medición.

No se consideran duplicadas dos mediciones únicamente porque posean la misma concentración;
deben coincidir también el informe, punto, fecha, hora, contaminante, ubicación y valor.

In [ ]:
duplicados = con.execute("""
SELECT
    SUM(cantidad - 1) AS duplicados_exactos

FROM (
    SELECT
        numero_informe,
        nombre_evaluacion,
        punto,
        este,
        norte,
        altitud,
        zona,
        datum,
        fecha,
        hora,
        contaminante,
        unidad,
        valor,

        COUNT(*) AS cantidad

    FROM etapa_espacial

    GROUP BY
        numero_informe,
        nombre_evaluacion,
        punto,
        este,
        norte,
        altitud,
        zona,
        datum,
        fecha,
        hora,
        contaminante,
        unidad,
        valor

    HAVING COUNT(*) > 1
);
""").df()

display(duplicados)

## 15. Eliminación de duplicados exactos

Una vez cuantificados los duplicados, se conserva una única instancia de las observaciones que son completamente idénticas según las variables seleccionadas.

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW etapa_sin_duplicados AS

SELECT DISTINCT
    numero_informe,
    nombre_evaluacion,

    punto,

    este,
    norte,
    altitud,
    zona,
    datum,

    fecha,
    hora,

    contaminante,
    unidad,
    valor

FROM etapa_espacial;
""")

In [ ]:
TOTAL_SIN_DUPLICADOS = con.execute("""
SELECT COUNT(*)
FROM etapa_sin_duplicados
""").fetchone()[0]

print(f"Antes de eliminar duplicados: {TOTAL_ESPACIAL:,}")
print(f"Después: {TOTAL_SIN_DUPLICADOS:,}")
print(f"Duplicados eliminados: {TOTAL_ESPACIAL - TOTAL_SIN_DUPLICADOS:,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Antes de eliminar duplicados: 1,293,176
Después: 1,246,271
Duplicados eliminados: 46,905


## 16. Dataset limpio

Se genera la vista final con las variables necesarias para conservar la trazabilidad de las mediciones y permitir su posterior preparación para Machine Learning.

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW eas_aire_limpio AS

SELECT
    numero_informe,
    nombre_evaluacion,

    punto,

    este,
    norte,
    altitud,
    zona,
    datum,

    fecha,
    hora,

    contaminante,
    valor AS concentracion,
    unidad

FROM etapa_sin_duplicados;
""")

## 17. Verificación del dataset limpio

In [ ]:
muestra_limpia = con.execute("""
SELECT *
FROM eas_aire_limpio
LIMIT 20
""").df()

display(muestra_limpia)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,numero_informe,nombre_evaluacion,punto,este,norte,altitud,zona,datum,fecha,hora,contaminante,concentracion,unidad
0,REAS-006-2024-STEC,Calidad del aire en el ámbito de la zona indus...,CA-CH-03,767988.0,8992825.0,15.0,17,WGS84,2023-01-01,02:00:00,SO2,40.64,µg/m³
1,REAS-006-2024-STEC,Calidad del aire en el ámbito de la zona indus...,CA-CH-03,767988.0,8992825.0,15.0,17,WGS84,2023-01-01,03:00:00,SO2,36.76,µg/m³
2,REAS-006-2024-STEC,Calidad del aire en el ámbito de la zona indus...,CA-CH-03,767988.0,8992825.0,15.0,17,WGS84,2023-01-01,05:00:00,SO2,20.83,µg/m³
3,REAS-006-2024-STEC,Calidad del aire en el ámbito de la zona indus...,CA-CH-03,767988.0,8992825.0,15.0,17,WGS84,2023-01-01,06:00:00,SO2,33.41,µg/m³
4,REAS-006-2024-STEC,Calidad del aire en el ámbito de la zona indus...,CA-CH-03,767988.0,8992825.0,15.0,17,WGS84,2023-01-01,11:00:00,SO2,35.95,µg/m³
5,REAS-006-2024-STEC,Calidad del aire en el ámbito de la zona indus...,CA-CH-03,767988.0,8992825.0,15.0,17,WGS84,2023-01-01,12:00:00,SO2,25.20,µg/m³
6,REAS-006-2024-STEC,Calidad del aire en el ámbito de la zona indus...,CA-CH-03,767988.0,8992825.0,15.0,17,WGS84,2023-01-01,14:00:00,SO2,13.83,µg/m³
7,REAS-006-2024-STEC,Calidad del aire en el ámbito de la zona indus...,CA-CH-03,767988.0,8992825.0,15.0,17,WGS84,2023-01-01,17:00:00,SO2,11.48,µg/m³
8,REAS-006-2024-STEC,Calidad del aire en el ámbito de la zona indus...,CA-CH-03,767988.0,8992825.0,15.0,17,WGS84,2023-01-01,21:00:00,SO2,10.87,µg/m³
9,REAS-006-2024-STEC,Calidad del aire en el ámbito de la zona indus...,CA-CH-03,767988.0,8992825.0,15.0,17,WGS84,2023-01-01,22:00:00,SO2,10.27,µg/m³


In [ ]:
esquema_limpio = con.execute("""
DESCRIBE eas_aire_limpio
""").df()

display(esquema_limpio)

,column_name,column_type,null,key,default,extra
0,numero_informe,VARCHAR,YES,None,None,None
1,nombre_evaluacion,VARCHAR,YES,None,None,None
2,punto,VARCHAR,YES,None,None,None
3,este,DOUBLE,YES,None,None,None
4,norte,DOUBLE,YES,None,None,None
5,altitud,DOUBLE,YES,None,None,None
6,zona,INTEGER,YES,None,None,None
7,datum,VARCHAR,YES,None,None,None
8,fecha,DATE,YES,None,None,None
9,hora,TIME,YES,None,None,None


## 18. Distribución final de contaminantes

Se verifica cuántos registros permanecen disponibles para cada contaminante después
de las principales etapas de limpieza.

También se revisan los valores mínimo, promedio y máximo como una primera evaluación
de la distribución de las concentraciones.

In [ ]:
distribucion_final = con.execute("""
SELECT
    contaminante,
    COUNT(*) AS registros,

    MIN(concentracion) AS minimo,
    AVG(concentracion) AS promedio,
    MAX(concentracion) AS maximo

FROM eas_aire_limpio

GROUP BY contaminante
ORDER BY registros DESC
""").df()

display(distribucion_final)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,contaminante,registros,minimo,promedio,maximo
0,PM10,461641,0.01,42.567106,6024.55
1,PM2.5,413499,0.00,20.799127,1256.30
2,SO2,212557,0.47,27.138638,5654.69
3,NO2,81804,1.07,13.113193,993.73
4,CO,76770,27.67,292.744725,1705.06


## 19. Distribución temporal final

Se comprueba que el conjunto limpio conserve observaciones correspondientes a los
tres años seleccionados: 2022, 2023 y 2024.

In [ ]:
por_anio = con.execute("""
SELECT
    YEAR(fecha) AS anio,
    COUNT(*) AS registros

FROM eas_aire_limpio

GROUP BY YEAR(fecha)
ORDER BY anio
""").df()

display(por_anio)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,anio,registros
0,2022,411633
1,2023,435356
2,2024,399282


## 20. Análisis de distribución de concentraciones

Antes de finalizar el proceso de limpieza se analiza la distribución de las concentraciones
para identificar valores extremos.

Los valores elevados no son eliminados automáticamente, ya que podrían representar eventos
reales de contaminación. Primero se evalúa su frecuencia y posición dentro de la distribución
de cada contaminante.

In [ ]:
estadisticas = con.execute("""
SELECT
    contaminante,

    COUNT(*) AS registros,

    MIN(concentracion) AS minimo,

    QUANTILE_CONT(concentracion, 0.25) AS q1,
    MEDIAN(concentracion) AS mediana,
    QUANTILE_CONT(concentracion, 0.75) AS q3,

    QUANTILE_CONT(concentracion, 0.90) AS p90,
    QUANTILE_CONT(concentracion, 0.95) AS p95,
    QUANTILE_CONT(concentracion, 0.99) AS p99,

    AVG(concentracion) AS promedio,
    MAX(concentracion) AS maximo

FROM eas_aire_limpio

GROUP BY contaminante
ORDER BY contaminante;
""").df()

display(estadisticas)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,contaminante,registros,minimo,q1,mediana,q3,p90,p95,p99,promedio,maximo
0,CO,76770,27.67,246.19,281.70,324.83,378.580,433.6555,638.6313,292.744725,1705.06
1,NO2,81804,1.07,4.76,7.16,12.09,22.444,50.7000,88.3185,13.113193,993.73
2,PM10,461641,0.01,12.54,24.20,49.83,102.120,146.7900,250.5660,42.567106,6024.55
3,PM2.5,413499,0.00,6.70,11.85,21.66,47.880,79.5000,139.7400,20.799127,1256.30
4,SO2,212557,0.47,7.60,9.51,19.70,66.310,111.9560,259.5232,27.138638,5654.69


## 21. Revisión de las concentraciones más elevadas

Se inspeccionan las observaciones con mayores concentraciones dentro de cada contaminante.

El objetivo es determinar si los valores elevados corresponden a registros aislados,
errores evidentes o posibles eventos reales registrados en determinados puntos y fechas.

In [ ]:
top_por_contaminante = con.execute("""
WITH ranking AS (
    SELECT
        numero_informe,
        nombre_evaluacion,
        punto,
        fecha,
        hora,
        contaminante,
        concentracion,
        unidad,

        ROW_NUMBER() OVER (
            PARTITION BY contaminante
            ORDER BY concentracion DESC
        ) AS posicion

    FROM eas_aire_limpio
)

SELECT *
FROM ranking

WHERE posicion <= 20

ORDER BY contaminante, posicion;
""").df()

display(top_por_contaminante)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,numero_informe,nombre_evaluacion,punto,fecha,hora,contaminante,concentracion,unidad,posicion
0,REAS-003-2024-STEC,Calidad del aire en el ámbito de influencia de...,CA-PISCO-03,2023-06-02,04:00:00,CO,1705.06,µg/m³,1
1,REAS-003-2024-STEC,Calidad del aire en el ámbito de influencia de...,CA-PISCO-03,2023-06-04,01:00:00,CO,1699.72,µg/m³,2
2,REAS-003-2024-STEC,Calidad del aire en el ámbito de influencia de...,CA-PISCO-03,2023-06-04,00:00:00,CO,1407.94,µg/m³,3
3,REAS-003-2024-STEC,Calidad del aire en el ámbito de influencia de...,CA-PISCO-03,2023-08-09,07:00:00,CO,1328.82,µg/m³,4
4,REAS-003-2024-STEC,Calidad del aire en el ámbito de influencia de...,CA-PISCO-03,2023-08-08,22:00:00,CO,1263.26,µg/m³,5
...,...,...,...,...,...,...,...,...,...
95,REAS-012-2025-STEC,Calidad del aire en el área de influencia del ...,CA-CC-01,2024-12-08,<NA>,SO2,1451.09,µg/m³,16
96,REAS-012-2025-STEC,Calidad del aire en el área de influencia del ...,CA-CC-01,2024-07-30,<NA>,SO2,1441.37,µg/m³,17
97,REAS-012-2025-STEC,Calidad del aire en el área de influencia del ...,CA-CC-01,2024-09-09,<NA>,SO2,1439.06,µg/m³,18
98,REPORTE N° 00007-2022-OEFA/DEAM-STEC,Calidad del aire en el ámbito de la refinería ...,CA-RZC-02,2022-04-15,<NA>,SO2,1432.96,µg/m³,19


## 22. Identificación estadística de valores extremos

Se utiliza el rango intercuartílico (IQR) para identificar observaciones alejadas de la
distribución típica de cada contaminante.

In [ ]:
outliers_iqr = con.execute("""
WITH limites AS (
    SELECT
        contaminante,

        QUANTILE_CONT(concentracion, 0.25) AS q1,
        QUANTILE_CONT(concentracion, 0.75) AS q3

    FROM eas_aire_limpio

    GROUP BY contaminante
),

datos AS (
    SELECT
        e.*,
        l.q1,
        l.q3,
        (l.q3 - l.q1) AS iqr

    FROM eas_aire_limpio e

    JOIN limites l
      ON e.contaminante = l.contaminante
)

SELECT
    contaminante,

    COUNT(*) AS registros,

    COUNT(*) FILTER (
        WHERE concentracion > q3 + 1.5 * iqr
    ) AS valores_extremos,

    ROUND(
        100.0 *
        COUNT(*) FILTER (
            WHERE concentracion > q3 + 1.5 * iqr
        )
        / COUNT(*),
        2
    ) AS porcentaje_extremos

FROM datos

GROUP BY contaminante
ORDER BY contaminante;
""").df()

display(outliers_iqr)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,contaminante,registros,valores_extremos,porcentaje_extremos
0,CO,76770,3467,4.52
1,NO2,81804,7905,9.66
2,PM10,461641,43665,9.46
3,PM2.5,413499,45504,11.00
4,SO2,212557,34184,16.08


## 23. Distribución de valores elevados por punto de monitoreo

Se analiza la distribución espacial de las observaciones superiores al percentil 99 de cada
contaminante.

Si las concentraciones elevadas aparecen repetidamente en determinados puntos y periodos, pueden corresponder a eventos ambientales reales y no necesariamente a errores del dataset.

In [ ]:
extremos_por_punto = con.execute("""
WITH limites AS (
    SELECT
        contaminante,
        QUANTILE_CONT(concentracion, 0.99) AS p99

    FROM eas_aire_limpio

    GROUP BY contaminante
)

SELECT
    e.contaminante,
    e.punto,

    COUNT(*) AS registros_sobre_p99,

    MIN(e.fecha) AS primera_fecha,
    MAX(e.fecha) AS ultima_fecha,

    MAX(e.concentracion) AS maximo

FROM eas_aire_limpio e

JOIN limites l
  ON e.contaminante = l.contaminante

WHERE e.concentracion > l.p99

GROUP BY
    e.contaminante,
    e.punto

ORDER BY
    e.contaminante,
    registros_sobre_p99 DESC;
""").df()

display(extremos_por_punto.head(100))